# My LI Model Inference Demo
仿照 ml_model_inference_demo.ipynb，但使用我们自己训练的模型。

In [ ]:
# ===== 配置区 =====
import os
JAX_CFD_ROOT = '/jumbo/yaoqingyang/yuxin/JAX-CFD'

# 选择要评估的模型目录（改这里切换不同训练结果）
MODEL_DIR = os.path.join(JAX_CFD_ROOT, 'model_fixed_1ep')  # 1epoch
# MODEL_DIR = os.path.join(JAX_CFD_ROOT, 'model_fixed_10ep')  # 10epoch
# MODEL_DIR = os.path.join(JAX_CFD_ROOT, 'model_fixed_100ep')  # 100epoch

# 评估用的参考数据集
EVAL_NC = os.path.join(JAX_CFD_ROOT,
    'content/kolmogorov_re_1000/long_eval_2048x2048_64x64.nc')

# 选择 GPU（避免和训练进程冲突）
os.environ['CUDA_VISIBLE_DEVICES'] = '4'
os.environ['JAX_PLATFORMS'] = 'cuda'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['HAIKU_FLATMAPPING'] = '0'
os.environ['LD_LIBRARY_PATH'] = (
    '/jumbo/yaoqingyang/batman/miniconda3/envs/cfd-gpu/lib:'
    + os.environ.get('LD_LIBRARY_PATH', '')
)

In [ ]:
import sys, warnings
warnings.simplefilter('ignore')
sys.path.insert(0, JAX_CFD_ROOT)
sys.path.insert(0, os.path.join(JAX_CFD_ROOT, 'jax-cfd'))

import functools, pickle
import gin
import jax
import jax.numpy as jnp
import numpy as np
import haiku as hk
import xarray
import matplotlib.pyplot as plt
from flax.training import checkpoints

import jax_cfd.base as cfd
import jax_cfd.data as cfd_data
import jax_cfd.ml as ml

# 我们的自定义模块
import my_model_builder
import my_equations
import my_encoders
import my_decoders
import my_interpolations
import my_towers
import my_advections

model_builder = ml.model_builder
physics_specifications = ml.physics_specifications
print('JAX devices:', jax.devices())

## 1. 加载评估数据集

In [ ]:
# 加载参考数据集（2048x2048 DNS → 64x64）
reference_ds = xarray.open_dataset(EVAL_NC, chunks={'time': '100MB'})
grid = cfd_data.xarray_utils.grid_from_attrs(reference_ds.attrs)
print('Grid:', grid)
print('Reference dataset:', dict(reference_ds.dims))

In [ ]:
# 选取初始条件和轨迹长度
sample_id = 0
time_id = 0
length = 200     # 轨迹步数（参考数据的时间步，dt ≈ 0.07s）
inner_steps = 10  # 每步模型推理包含的内部步数（匹配 predict_steps=16 → 大约每 dt_ref 需要 10 个 dt_model 步）

initial_conditions = tuple(
    reference_ds[v].isel(
        sample=slice(sample_id, sample_id + 1),
        time=slice(time_id, time_id + 1)
    ).values
    for v in cfd_data.xarray_utils.XR_VELOCITY_NAMES[:grid.ndim]
)

target_ds = reference_ds.isel(
    sample=slice(sample_id, sample_id + 1),
    time=slice(time_id, time_id + length))

print('Initial conditions shapes:', [x.shape for x in initial_conditions])
print('Target dataset:', dict(target_ds.dims))

## 2. 加载我们训练的模型 checkpoint

In [ ]:
# 加载最新 checkpoint
print('Loading checkpoint from:', MODEL_DIR)
ckpt_state = checkpoints.restore_checkpoint(MODEL_DIR, target=None)
params = ckpt_state['params']
print('Checkpoint step:', ckpt_state.get('step', 'unknown'))

# 打印参数结构
print('\nParam keys (first 5):')
for k in list(params.keys())[:5]:
    shapes = {kk: vv.shape for kk, vv in params[k].items()}
    print(f'  {k}: {shapes}')

## 3. 构建模型（用我们的 gin 配置）

In [ ]:
def strip_imports(s):
    return '\n'.join(l for l in s.splitlines() if not l.startswith('import'))

# 解析我们的 gin 配置
gin.clear_config()
gin.parse_config_files_and_bindings(
    [
        os.path.join(JAX_CFD_ROOT, 'models/configs/official_li_config.gin'),
        os.path.join(JAX_CFD_ROOT, 'models/configs/kolmogorov_forcing.gin'),
    ],
    [
        'fixed_scale.rescaled_one = 0.2',
        'my_forward_tower_factory.num_hidden_channels = 128',
        'my_forward_tower_factory.num_hidden_layers = 6',
        'MyFusedLearnedInterpolation.pattern = "simple"',
    ]
)
# 加载物理参数（来自训练数据集的属性）
train_ds = xarray.open_dataset(
    os.path.join(JAX_CFD_ROOT,
                 'content/kolmogorov_re_1000/train_2048x2048_64x64.nc'))
gin.parse_config(strip_imports(train_ds.attrs['physics_config_str']))

dt = float(train_ds.attrs['stable_time_step'])
print(f'dt = {dt}')

physics_specs_obj = physics_specifications.get_physics_specs()
model_cls = model_builder.get_model_cls(grid, dt, physics_specs_obj)
print('Model class:', model_cls)

In [ ]:
# 构建推理函数
def compute_trajectory_fwd(x):
    solver = model_cls()
    x = solver.encode(x)
    final, trajectory = solver.trajectory(
        x, length, inner_steps, start_with_input=True,
        post_process_fn=solver.decode)
    return trajectory

model = hk.without_apply_rng(hk.transform(compute_trajectory_fwd))
trajectory_fn = functools.partial(model.apply, params)
trajectory_fn = jax.vmap(trajectory_fn)  # 支持 batch
print('Trajectory function built.')

## 4. 运行推理

In [ ]:
print('Running inference...')
prediction = trajectory_fn(initial_conditions)
prediction_ds = cfd_data.xarray_utils.velocity_trajectory_to_xarray(
    prediction, grid, samples=True)

# 对齐时间坐标
prediction_ds.coords['x'] = target_ds.coords['x']
prediction_ds.coords['y'] = target_ds.coords['y']
prediction_ds.coords['time'] = target_ds.coords['time']

print('Prediction shape:', dict(prediction_ds.dims))

## 5. 计算评估指标（相关性 & 能量谱）

In [ ]:
# 加载不同分辨率 baseline（和原始 demo 一样）
base_path = os.path.join(JAX_CFD_ROOT, 'content/kolmogorov_re_1000')
datasets = {}
for res in [64, 128, 256, 512, 1024, 2048]:
    nc_path = os.path.join(base_path, f'long_eval_{res}x{res}_64x64.nc')
    if os.path.exists(nc_path):
        datasets[f'baseline_{res}x{res}'] = xarray.open_dataset(
            nc_path, chunks={'time': '100MB'}).isel(
            sample=slice(sample_id, sample_id + 1),
            time=slice(time_id, time_id + length))
    else:
        print(f'Missing: {nc_path}')

# 加入我们模型的预测
datasets['Our LI Model'] = prediction_ds

print('Available datasets:', list(datasets.keys()))

In [ ]:
import xarray

summary = xarray.concat([
    cfd_data.evaluation.compute_summary_dataset(ds, target_ds)
    for ds in datasets.values()
], dim='model')
summary.coords['model'] = list(datasets.keys())

correlation = summary.vorticity_correlation.compute()
spectrum    = summary.energy_spectrum_mean.mean('time').compute()

## 6. 可视化

In [ ]:
import seaborn

n_baselines = len(datasets) - 1
baseline_palette = seaborn.color_palette('YlGnBu', n_colors=n_baselines + 1)[1:]
model_color = seaborn.xkcd_palette(['burnt orange'])
palette = baseline_palette + model_color

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 涡度相关性
ax = axes[0]
for color, model_name in zip(palette, summary['model'].data):
    style = '-' if 'baseline' in model_name else '--'
    lw = 2 if 'baseline' in model_name else 3
    correlation.sel(model=model_name).plot.line(
        ax=ax, color=color, linestyle=style, label=model_name, linewidth=lw)
ax.axhline(y=0.95, color='gray', linestyle=':', linewidth=1)
ax.set_title('Vorticity Correlation')
ax.set_xlabel('Time')
ax.set_ylabel('Correlation')
ax.legend(fontsize=7)

# 能量谱
ax = axes[1]
for color, model_name in zip(palette, summary['model'].data):
    style = '-' if 'baseline' in model_name else '--'
    lw = 2 if 'baseline' in model_name else 3
    spectrum.sel(model=model_name).plot.line(
        ax=ax, color=color, linestyle=style, label=model_name, linewidth=lw)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_title('Energy Spectrum × k⁵')
ax.set_xlabel('k')
ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'inference_demo_eval.png'), dpi=120)
plt.show()
print('Saved to', os.path.join(MODEL_DIR, 'inference_demo_eval.png'))

In [ ]:
# ===== 涡度场可视化 =====
def vorticity(ds, t_idx):
    u = ds.u.isel(sample=0, time=t_idx).values
    v = ds.v.isel(sample=0, time=t_idx).values
    dx = float(ds.x.values[1] - ds.x.values[0])
    return np.gradient(v, dx, axis=0) - np.gradient(u, dx, axis=1)

t_indices = [0, 20, 50, 100]
show_models = ['baseline_2048x2048', 'Our LI Model']

fig, axes = plt.subplots(len(show_models), len(t_indices),
                         figsize=(4 * len(t_indices), 3.5 * len(show_models)))
for row, mname in enumerate(show_models):
    ds = datasets[mname]
    for col, tidx in enumerate(t_indices):
        w = vorticity(ds, tidx)
        vm = np.percentile(np.abs(w), 98)
        axes[row, col].imshow(w.T, origin='lower', cmap='RdBu_r',
                              vmin=-vm, vmax=vm)
        axes[row, col].set_title(f'{mname}\nt={float(ds.time.values[tidx]):.2f}')
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'inference_demo_vorticity.png'), dpi=120)
plt.show()
print('Saved.')

## 说明
- **蓝色系曲线**：不同分辨率的 DNS baseline（64×64 最差，2048×2048 最好）
- **橙色虚线**：我们的 LI 模型
- 涡度相关性越高、持续越久越好（理想是贴近 2048×2048 baseline）
- 能量谱在高波数段越平越好
- 如需切换模型，修改顶部 `MODEL_DIR` 变量重新运行即可